# Spatial Relationships Graph with topologic_fast

This notebook demonstrates how to create spatial relationship graphs from building geometry.
It covers:

1. Creating building elements (walls, slabs, doors, windows)
2. Computing spatial relationships between elements
3. Building a graph of spatial relationships
4. Visualizing the spatial graph with color-coded relationships

**Note:** Direct IFC file parsing is not yet implemented in topologic_fast.
This notebook demonstrates the spatial relationship concepts using programmatically created geometry.

## Import Required Libraries

In [ ]:
# Import topologic_fast
import topologic_fast as tf

# Import visualization libraries
import plotly.graph_objects as go
import numpy as np
from collections import defaultdict

print("topologic_fast imported successfully")

## Define Spatial Relationship Types

We define the DE-9IM based spatial relationships used in topological analysis.

In [ ]:
# Spatial relationship color mapping (based on topologicpy conventions)
RELATIONSHIP_COLORS = {
    "contains": "#FF0000",     # Red - A contains B
    "within": "#FF0000",       # Red - A is within B (inverse of contains)
    "covers": "#0000C8",       # Blue - A covers B
    "coveredBy": "#0000C8",    # Blue - A is covered by B
    "crosses": "#0098FF",      # Light blue - geometries cross
    "disjoint": "#2CFF96",     # Green - no overlap
    "equals": "#97FF00",       # Yellow-green - identical
    "overlaps": "#FFEA00",     # Yellow - partial overlap
    "touches": "#550E55",      # Purple - share boundary
    "near": "#AAAAAA",         # Gray - within threshold
    "intermediate": "#666666", # Dark gray
    "far": "#000000"           # Black - beyond threshold
}

# IFC type color mapping
IFC_TYPE_COLORS = {
    "IfcWall": "#FF6B6B",
    "IfcWallStandardCase": "#FF6B6B",
    "IfcSlab": "#4ECDC4",
    "IfcDoor": "#95E1D3",
    "IfcWindow": "#87CEEB",
    "IfcSpace": "#DDA0DD",
    "Unknown": "#CCCCCC"
}

print("Relationship types defined:")
for rel, color in RELATIONSHIP_COLORS.items():
    print(f"  - {rel}: {color}")

## Create Sample Building Elements

Since IFC parsing is not yet available, we create building elements programmatically.

In [ ]:
class BuildingElement:
    """A simple wrapper for building elements with IFC-like properties."""
    def __init__(self, geometry, ifc_type, name):
        self.geometry = geometry  # topologic_fast topology
        self.ifc_type = ifc_type
        self.name = name
    
    def bounding_box(self):
        """Get the axis-aligned bounding box."""
        if hasattr(self.geometry, 'BoundingBox'):
            return self.geometry.BoundingBox()
        elif hasattr(self.geometry, 'Vertices'):
            vertices = self.geometry.Vertices()
            coords = [v.Coordinates() for v in vertices]
            xs = [c[0] for c in coords]
            ys = [c[1] for c in coords]
            zs = [c[2] for c in coords]
            return ((min(xs), min(ys), min(zs)), (max(xs), max(ys), max(zs)))
        return ((0, 0, 0), (1, 1, 1))
    
    def center(self):
        """Get the center of mass."""
        if hasattr(self.geometry, 'CenterOfMass'):
            return self.geometry.CenterOfMass()
        bbox = self.bounding_box()
        return (
            (bbox[0][0] + bbox[1][0]) / 2,
            (bbox[0][1] + bbox[1][1]) / 2,
            (bbox[0][2] + bbox[1][2]) / 2
        )

# Create a simple building structure
elements = []

# Floor slab
floor_slab = tf.Cell.Box(0, 0, 0, 10, 10, 0.3)
elements.append(BuildingElement(floor_slab, "IfcSlab", "Floor_Slab_01"))

# Walls (4 walls around the perimeter)
wall_thickness = 0.2
wall_height = 3.0

# North wall
north_wall = tf.Cell.Box(0, 10 - wall_thickness, 0.3, 10, wall_thickness, wall_height)
elements.append(BuildingElement(north_wall, "IfcWall", "Wall_North"))

# South wall
south_wall = tf.Cell.Box(0, 0, 0.3, 10, wall_thickness, wall_height)
elements.append(BuildingElement(south_wall, "IfcWall", "Wall_South"))

# East wall
east_wall = tf.Cell.Box(10 - wall_thickness, 0, 0.3, wall_thickness, 10, wall_height)
elements.append(BuildingElement(east_wall, "IfcWall", "Wall_East"))

# West wall
west_wall = tf.Cell.Box(0, 0, 0.3, wall_thickness, 10, wall_height)
elements.append(BuildingElement(west_wall, "IfcWall", "Wall_West"))

# Door in south wall
door = tf.Cell.Box(4, 0, 0.3, 1.2, wall_thickness, 2.1)
elements.append(BuildingElement(door, "IfcDoor", "Door_Main"))

# Window in east wall
window = tf.Cell.Box(10 - wall_thickness, 3, 1.0, wall_thickness, 2, 1.5)
elements.append(BuildingElement(window, "IfcWindow", "Window_East"))

# Ceiling/Roof slab
roof_slab = tf.Cell.Box(0, 0, 0.3 + wall_height, 10, 10, 0.3)
elements.append(BuildingElement(roof_slab, "IfcSlab", "Roof_Slab_01"))

print(f"Created {len(elements)} building elements:")
for elem in elements:
    print(f"  - {elem.name} ({elem.ifc_type})")

## Compute Spatial Relationships

We compute spatial relationships between elements using bounding box intersection and proximity tests.

In [ ]:
def boxes_intersect(bbox1, bbox2, tolerance=0.01):
    """Check if two bounding boxes intersect (with tolerance)."""
    min1, max1 = bbox1
    min2, max2 = bbox2
    
    # Check for separation along each axis
    for i in range(3):
        if max1[i] + tolerance < min2[i] or max2[i] + tolerance < min1[i]:
            return False
    return True

def boxes_touch(bbox1, bbox2, tolerance=0.05):
    """Check if two bounding boxes touch (share a boundary)."""
    min1, max1 = bbox1
    min2, max2 = bbox2
    
    # Check if they touch on any face
    for i in range(3):
        if abs(max1[i] - min2[i]) < tolerance or abs(max2[i] - min1[i]) < tolerance:
            # Check overlap in other dimensions
            other_dims = [j for j in range(3) if j != i]
            overlaps = all(
                not (max1[j] < min2[j] - tolerance or max2[j] < min1[j] - tolerance)
                for j in other_dims
            )
            if overlaps:
                return True
    return False

def box_contains(bbox_outer, bbox_inner, tolerance=0.01):
    """Check if outer box contains inner box."""
    min1, max1 = bbox_outer
    min2, max2 = bbox_inner
    
    for i in range(3):
        if min2[i] < min1[i] - tolerance or max2[i] > max1[i] + tolerance:
            return False
    return True

def compute_relationship(elem1, elem2, tolerance=0.05):
    """
    Compute spatial relationship between two building elements.
    
    Returns: relationship type string
    """
    bbox1 = elem1.bounding_box()
    bbox2 = elem2.bounding_box()
    
    # Check various relationships
    if box_contains(bbox1, bbox2):
        return "contains"
    elif box_contains(bbox2, bbox1):
        return "within"
    elif boxes_intersect(bbox1, bbox2, tolerance=0):
        return "overlaps"
    elif boxes_touch(bbox1, bbox2, tolerance):
        return "touches"
    else:
        # Calculate distance between centers
        c1 = elem1.center()
        c2 = elem2.center()
        dist = np.sqrt(sum((a - b) ** 2 for a, b in zip(c1, c2)))
        
        if dist < 2.0:
            return "near"
        elif dist < 5.0:
            return "intermediate"
        else:
            return "far"

# Compute relationships between all element pairs
relationships = []
include_rels = ["overlaps", "touches", "within", "contains"]

for i, elem1 in enumerate(elements):
    for j, elem2 in enumerate(elements):
        if i >= j:  # Avoid duplicates and self-comparison
            continue
        
        rel = compute_relationship(elem1, elem2)
        
        if rel in include_rels:
            relationships.append({
                'elem1': elem1,
                'elem2': elem2,
                'relationship': rel,
                'index1': i,
                'index2': j
            })

print(f"\nFound {len(relationships)} spatial relationships:")
for r in relationships:
    print(f"  {r['elem1'].name} <-{r['relationship']}-> {r['elem2'].name}")

## Build a Graph from Spatial Relationships

Create a topologic_fast Graph representing the spatial relationships.

In [ ]:
# Create vertices at element centers
vertices = []
vertex_to_element = {}

for i, elem in enumerate(elements):
    center = elem.center()
    v = tf.Vertex.ByCoordinates(center[0], center[1], center[2])
    vertices.append(v)
    vertex_to_element[i] = elem

# Create edges for relationships
edges = []
edge_relationships = []

for r in relationships:
    v1 = vertices[r['index1']]
    v2 = vertices[r['index2']]
    edge = tf.Edge.ByStartVertexEndVertex(v1, v2)
    edges.append(edge)
    edge_relationships.append(r['relationship'])

# Create the graph
graph = tf.Graph.ByVerticesEdges(vertices, edges)

print(f"\nGraph created:")
print(f"  Vertices: {len(graph.Vertices())}")
print(f"  Edges: {len(graph.Edges())}")

## Analyze Graph Properties

In [ ]:
# Compute graph metrics
print("Graph Analysis:")
print(f"  Density: {graph.Density():.3f}")
print(f"  Diameter: {graph.Diameter()}")
print(f"  Is Bipartite: {graph.IsBipartite()}")
print(f"  Is Complete: {graph.IsComplete()}")

# Degree sequence
degree_seq = graph.DegreeSequence()
print(f"\nDegree Sequence: {degree_seq}")

# Element connectivity
print("\nElement Connectivity (degree):")
for i, v in enumerate(vertices):
    degree = degree_seq[i] if i < len(degree_seq) else 0
    elem = vertex_to_element[i]
    print(f"  {elem.name}: {degree} connections")

## Visualize the Spatial Relationships Graph

Create an interactive 3D visualization showing both the building geometry and the spatial relationship graph.

In [ ]:
def create_mesh_from_cell(cell):
    """Create mesh data for Plotly from a Cell."""
    faces = cell.Faces()
    all_x, all_y, all_z = [], [], []
    all_i, all_j, all_k = [], [], []
    
    for face in faces:
        vertices = face.Vertices()
        if len(vertices) < 3:
            continue
        
        coords = [v.Coordinates() for v in vertices]
        offset = len(all_x)
        
        # Add vertices
        for c in coords:
            all_x.append(c[0])
            all_y.append(c[1])
            all_z.append(c[2])
        
        # Fan triangulation
        for idx in range(1, len(coords) - 1):
            all_i.append(offset)
            all_j.append(offset + idx)
            all_k.append(offset + idx + 1)
    
    return (all_x, all_y, all_z), (all_i, all_j, all_k)

def visualize_spatial_graph(elements, relationships, vertices):
    """Create interactive 3D visualization of building and spatial graph."""
    fig = go.Figure()
    
    # Add building elements
    for elem in elements:
        coords, indices = create_mesh_from_cell(elem.geometry)
        if not coords[0]:
            continue
        
        color = IFC_TYPE_COLORS.get(elem.ifc_type, IFC_TYPE_COLORS["Unknown"])
        
        fig.add_trace(go.Mesh3d(
            x=coords[0], y=coords[1], z=coords[2],
            i=indices[0], j=indices[1], k=indices[2],
            color=color,
            opacity=0.2,
            name=f"{elem.name} ({elem.ifc_type})",
            flatshading=True,
            showlegend=True
        ))
    
    # Add graph edges (spatial relationships)
    for r in relationships:
        c1 = r['elem1'].center()
        c2 = r['elem2'].center()
        rel = r['relationship']
        color = RELATIONSHIP_COLORS.get(rel, "#000000")
        
        fig.add_trace(go.Scatter3d(
            x=[c1[0], c2[0]],
            y=[c1[1], c2[1]],
            z=[c1[2], c2[2]],
            mode='lines',
            line=dict(color=color, width=4),
            name=f"{rel}",
            hovertext=f"{r['elem1'].name} <-> {r['elem2'].name}: {rel}",
            showlegend=False
        ))
    
    # Add graph vertices (element centers)
    centers = [elem.center() for elem in elements]
    colors = [IFC_TYPE_COLORS.get(elem.ifc_type, IFC_TYPE_COLORS["Unknown"]) for elem in elements]
    
    fig.add_trace(go.Scatter3d(
        x=[c[0] for c in centers],
        y=[c[1] for c in centers],
        z=[c[2] for c in centers],
        mode='markers+text',
        marker=dict(
            size=12,
            color=colors,
            line=dict(color='black', width=2)
        ),
        text=[elem.name for elem in elements],
        textposition='top center',
        name='Element Centers',
        showlegend=True
    ))
    
    # Update layout
    fig.update_layout(
        title='Spatial Relationships Graph',
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.0)
            )
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    return fig

# Create and show visualization
fig = visualize_spatial_graph(elements, relationships, vertices)
fig.show()

## Relationship Legend

In [ ]:
# Create a legend showing relationship colors
fig_legend = go.Figure()

# Count relationships by type
rel_counts = defaultdict(int)
for r in relationships:
    rel_counts[r['relationship']] += 1

y_pos = list(range(len(RELATIONSHIP_COLORS)))
rel_types = list(RELATIONSHIP_COLORS.keys())
colors = list(RELATIONSHIP_COLORS.values())

fig_legend.add_trace(go.Bar(
    y=rel_types,
    x=[rel_counts.get(r, 0) for r in rel_types],
    orientation='h',
    marker=dict(color=colors),
    text=[f"{rel_counts.get(r, 0)} occurrences" for r in rel_types],
    textposition='outside'
))

fig_legend.update_layout(
    title='Spatial Relationship Types',
    xaxis_title='Count',
    yaxis_title='Relationship Type',
    height=400,
    width=600
)

fig_legend.show()

## Query Relationships

Find elements based on their spatial relationships.

In [ ]:
def find_touching_elements(element_name, relationships):
    """Find all elements that touch a given element."""
    touching = []
    for r in relationships:
        if r['relationship'] == 'touches':
            if r['elem1'].name == element_name:
                touching.append(r['elem2'].name)
            elif r['elem2'].name == element_name:
                touching.append(r['elem1'].name)
    return touching

def find_overlapping_elements(element_name, relationships):
    """Find all elements that overlap with a given element."""
    overlapping = []
    for r in relationships:
        if r['relationship'] == 'overlaps':
            if r['elem1'].name == element_name:
                overlapping.append(r['elem2'].name)
            elif r['elem2'].name == element_name:
                overlapping.append(r['elem1'].name)
    return overlapping

# Example queries
print("Spatial Relationship Queries:")
print("-" * 40)

for elem in elements:
    touching = find_touching_elements(elem.name, relationships)
    overlapping = find_overlapping_elements(elem.name, relationships)
    
    if touching or overlapping:
        print(f"\n{elem.name} ({elem.ifc_type}):")
        if touching:
            print(f"  Touches: {', '.join(touching)}")
        if overlapping:
            print(f"  Overlaps: {', '.join(overlapping)}")

## Summary

This notebook demonstrated spatial relationship analysis using topologic_fast:

1. **Building Elements**: Created walls, slabs, doors, and windows as Cells
2. **Spatial Relationships**: Computed relationships (touches, overlaps, contains, within)
3. **Graph Construction**: Built a Graph from vertices at element centers with edges for relationships
4. **Graph Analysis**: Computed density, diameter, and degree sequence
5. **Visualization**: Created interactive 3D views with color-coded relationships

### Features Not Yet Implemented in topologic_fast

- `Topology.ByIFCPath()` - Import IFC files directly
- `Graph.BySpatialRelationships()` - Automatic spatial relationship computation
- DE-9IM relationship matrix computation
- Direct IFC property transfer to Dictionary

These features are planned for future releases.

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")